# Phase 4: WavLM Audio Deepfake Training
Run this notebook with a Colab GPU runtime. It downloads only the official train and development splits. The evaluation split remains untouched. The final checkpoint bundle is downloaded to your computer.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU first.'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone --branch agent/phase-4-wavlm-training https://github.com/KJSK-Koushik/Audio-Deepfake-Detection-using-Self-Supervised-Speech-Representations.git
%cd Audio-Deepfake-Detection-using-Self-Supervised-Speech-Representations

In [ ]:
!pip install -q 'transformers>=4.57,<4.58' 'safetensors>=0.4,<1' 'pyarrow>=17,<25' 'soundfile>=0.12,<1' 'scipy>=1.13,<2' 'scikit-learn>=1.5,<2' 'pyyaml>=6,<7' 'tqdm>=4.66,<5'

In [ ]:
!python -m scripts.download_asvspoof2019_hf --splits train dev

In [ ]:
from pathlib import Path

OUTPUT_DIR = '/content/wavlm_detector'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Best checkpoint will be saved temporarily to:', OUTPUT_DIR)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m scripts.train_wavlm --device cuda --epochs 3 --batch-size 1 --gradient-accumulation 16 --num-workers 2 --output-dir "{OUTPUT_DIR}"

In [ ]:
import json
from pathlib import Path

summary = json.loads((Path(OUTPUT_DIR) / 'training_summary.json').read_text())
print(json.dumps(summary, indent=2))

In [ ]:
import shutil

from google.colab import files

archive_path = shutil.make_archive('/content/wavlm_detector', 'zip', OUTPUT_DIR)
print('Downloading checkpoint bundle:', archive_path)
files.download(archive_path)